# L4b: Single Asset Geometric Brownian Motion Models
In this lecture, we introduce our first continuous-time model for asset prices: geometric Brownian motion (GBM). Last lecture, we used lattice models to compute the probability that a stock trade exceeds a target scaled net present value (NPV). We now develop the same calculation using a model whose future share price has a continuous distribution.

> __Learning Objectives:__
>
> By the end of this lecture, you will be able to:
>
> * **Describe and simulate geometric Brownian motion:** Explain the drift and random fluctuations in the GBM model, interpret its lognormal price distribution, and use its exact solution to simulate prices on a time grid.
> * **Estimate and interpret GBM parameters:** Estimate mean growth and volatility from historical growth rates, recover the GBM drift, and explain how log-price regression provides another estimate of mean growth.
> * **Assess GBM and calculate trade probabilities:** Compare the model with the stylized facts of observed growth rates and calculate the probability that a long stock position exceeds a target scaled NPV at a scheduled sale time.

The GBM model has an exact solution, which allows us to simulate possible price paths and calculate the target probability directly. We will estimate its parameters from historical data and examine which features of observed price changes it can explain.

Let's get started!

___


## Examples
Two examples connect the lecture’s formulas to historical data and trading decisions:

> [▶ Estimate a single-asset GBM](CHEME-5660-L4b-Example-Parameters-SAGBM-Fall-2026.ipynb). How do we build a GBM model from historical share prices, and how well does it describe the data? We estimate mean growth and volatility, recover the GBM drift, and compare two methods for estimating mean growth. We then simulate price paths and compare the model’s price distribution with an observed historical trajectory, examining where its assumptions limit the comparison.

> [▶ Evaluate an NPV-based GBM trade rule](CHEME-5660-L4b-Example-GBM-NPV-TradeRule-Fall-2026.ipynb). How likely is a stock trade to exceed our target scaled NPV at a scheduled sale time? We use the estimated GBM parameters, a holding period, and a benchmark rate to calculate this probability. We check the calculation using a target with a known probability and plot how the probability changes as we raise the target.

Optional extensions are collected near the end of the lecture.

___


## From Lattices to Continuous Prices

Last lecture, we used lattice models to describe possible future share prices
and compute the probability of exceeding a target scaled NPV. These models
advance in discrete time steps, with a finite set of possible price movements
at each step. Let's now consider how this description changes as we make the
time step smaller.

> __Idea:__ Suppose we keep the holding period fixed and divide it into more
> steps. As the time step shrinks, we also reduce the size of each price
> movement and adjust its probability. With appropriate scaling, we can retain
> a finite mean and variance for the log return over the holding period,
> even as the number of steps grows without bound.


For suitably scaled, independent binomial steps, the accumulated log return
$\ln(S_T/S_0)$ approaches a normal distribution. Since
$S_T=S_0\exp[\ln(S_T/S_0)]$, the terminal share price approaches a lognormal
distribution. The corresponding continuous-time price model is
__geometric Brownian motion (GBM)__.

We will introduce GBM directly through its stochastic differential equation,
which describes how the price changes through time. We can then use its
solution to return to our trading question: what is the probability that the
scaled NPV exceeds our target at the scheduled sale time?

___


## Single Asset Geometric Brownian Motion (GBM)

Geometric Brownian motion (GBM) describes a share price whose changes combine a
drift component and random fluctuations. Both contributions are proportional to
the current share price $S(t)$. Starting from an initial price $S(0)=S_0>0$,
the price obeys:

$$
\frac{dS(t)}{S(t)}=\mu\,dt+\sigma\,dW(t),
\qquad 0\leq t\leq T.
$$

Here, $\mu\in\mathbb R$ is the constant __drift__ parameter, $\sigma>0$ is the
constant __volatility__ parameter, and $dW(t)$ represents an increment of a
Wiener process. We measure time in years, so $\mu$ has units of
$\text{year}^{-1}$ and $\sigma$ has units of $\text{year}^{-1/2}$.

> __How do we interpret the model?__ The left-hand side describes the fractional
> change in share price. The drift term determines its expected change, while
> the Wiener term introduces random fluctuations whose magnitude is controlled
> by the volatility. Because these are fractional changes, the same model
> parameters produce larger dollar fluctuations when the share price is higher.

The drift $\mu$ differs from the continuously compounded growth rate $g$
computed from price data. We will develop their relationship using the GBM
solution. First, what is a Wiener process?

> __Wiener process:__
> 
> A Wiener process $\{W(t):0\leq t\leq T\}$ is a real-valued stochastic process with continuous sample paths and the following properties:
>
> * __Initial value:__ The process starts at zero: $W(0)=0$ with probability one.
> * __Independent increments:__ The increments $W(t_{1})-W(t_{0}),\ldots,W(t_{k})-W(t_{k-1})$ are independent for any $0\leq t_{0}<t_{1}<\cdots<t_{k}\leq T$.
> * __Normally distributed increments:__ For $0\leq s<t\leq T$, the increment law is $W(t)-W(s)\sim\mathcal N(0,t-s)$. The increment has mean zero and variance $t-s$. Equivalently, it has the same distribution as $\sqrt{t-s}Z$, where $Z\sim\mathcal N(0,1)$.

Over an interval $\Delta t$, the Wiener increment has standard deviation
$\sqrt{\Delta t}$. The volatility parameter scales this to
$\sigma\sqrt{\Delta t}$, the noise scale that appears in the GBM solution and
simulations.

One useful feature of GBM is that it has an exact analytical solution. Starting
from $S_0>0$, the share price at a fixed future time $T>0$ is given by:

$$
\boxed{
S_T=S_0\exp\!\left[
\left(\mu-\frac{\sigma^2}{2}\right)T+\sigma\sqrt{T}\,Z
\right],
\qquad Z\sim\mathcal N(0,1).
}
$$

Taking the logarithm gives:

$$
\ln\left(\frac{S_T}{S_0}\right)
=\left(\mu-\frac{\sigma^2}{2}\right)T+\sigma\sqrt{T}\,Z.
$$

Thus, the log return is normally distributed with mean $(\mu-\sigma^2/2)T$
and variance $\sigma^2T$. The share price itself is lognormally distributed
and remains strictly positive. Its expectation and variance are:

$$
\begin{aligned}
\mathbb E[S_T]&=S_0e^{\mu T},\\
\operatorname{Var}(S_T)&=S_0^2e^{2\mu T}\left(e^{\sigma^2T}-1\right).
\end{aligned}
$$

> __Drift and mean growth:__ The expected price grows at rate $\mu$, while the
> expected log return per unit time is $\mu-\sigma^2/2$. The term $\sigma^2/2$
> is the __half-variance correction__, which arises when applying
> [Itô’s lemma to the log price](CHEME-5660-L4b-GBM-Solution-Derivation-Fall-2026.ipynb).
> The median price is $S_0e^{(\mu-\sigma^2/2)T}$, below the expected price
> because the lognormal distribution is right skewed.

These expressions describe the price at a fixed future time. Let's now use
the solution to describe how prices evolve between successive observations
on a discrete time grid.

### Discrete-Time GBM Model

Market data record prices at discrete times, such as once per trading day.
Define the grid $t_j=j\Delta t$ for $j=0,1,\ldots,N$, where $\Delta t>0$
is measured in years and $T=N\Delta t$. Applying the GBM solution between
successive observations gives the __one-step transition__:

$$
\boxed{
S_{t_j}=S_{t_{j-1}}\exp\!\left[
\left(\mu-\frac{\sigma^2}{2}\right)\Delta t
+\sigma\sqrt{\Delta t}\,Z_j
\right].
}
$$

Here, $j=1,\ldots,N$, and the shocks $Z_1,\ldots,Z_N$ are independent
standard normal random variables. Their independence follows from the
independent Wiener increments over successive intervals. For constant $\mu$
and $\sigma$, this transition gives exact GBM prices at the grid points.

Let's use this one-step transition to generate multiple possible price paths
with Monte Carlo simulation.

### Monte Carlo Simulation of GBM

Monte Carlo simulation allows us to examine the range of price outcomes
predicted by GBM. Let's generate $M$ sample paths, each containing $N$ time
steps, using the following procedure.

__Initialize:__ Given the initial price $S_{0}$, arithmetic drift $\mu$, volatility $\sigma$, time step $\Delta{t}$, number of steps $N$, and number of sample paths $M$. Initialize an array $\mathbf{S}\in\mathbb{R}^{(N+1)\times M}$ to hold the simulated prices, one path per column, with rows indexed by $j = 0,1,\dots,N$.

For each sample path (each column of $\mathbf{S}$) __do:__
1. Set the price in the first row (grid point $j = 0$) to the initial price $S_{0}$.
2. For each time step $j = 1$ to $N$ __do:__
   - a. Generate a random sample from the standard normal distribution: $Z_{j} \sim \mathcal{N}(0,1)$
   - b. Compute the price at grid point $t_{j}$ from the price at $t_{j-1}$ using the one-step transition:
   $$
   S_{t_{j}} \gets S_{t_{j-1}} \cdot \exp\Biggl[\left(\mu-\frac{\sigma^{2}}{2}\right)\Delta{t} + \sigma\sqrt{\Delta{t}}\;{Z_{j}}\Biggr].
   $$

Every shock is a fresh independent draw, for every step and for every path. This process generates $M$ different price paths (columns) of $N$ steps each, every one a possible future trajectory of the asset price under the GBM model.

The simulation requires values for the drift $\mu$ and volatility $\sigma$.
Let's now estimate these parameters from historical share prices.

___


## Estimation of GBM Parameters

We can estimate the GBM parameters from historical share prices. To connect
these observations to the model, divide the one-step transition by
$S_{t_{j-1}}$, take the natural logarithm, and divide by $\Delta t$. This gives
the __one-step growth rate__ used in L3a:

$$
\boxed{
g_j\equiv\frac{1}{\Delta t}\ln\left(\frac{S_{t_j}}{S_{t_{j-1}}}\right)
=
\underbrace{\left(\mu-\frac{\sigma^2}{2}\right)}_{\text{mean growth }\mu_g}
+
\underbrace{\frac{\sigma}{\sqrt{\Delta t}}}_{\text{growth-rate std}}Z_j.
}
$$

Here, $j=1,\ldots,N$. Under constant-parameter GBM, the shocks $Z_j$ are
independent standard normal random variables, so growth rates over equally
spaced, non-overlapping intervals are independent and normally distributed.

> __Parameters:__
>
> * __Drift versus growth:__ The mean growth rate is $\mu_g=\mathbb E[g_j]=\mu-\sigma^2/2$. Thus, averaging observed growth rates estimates $\mu_g$. To recover the GBM drift $\mu$, we add the half-variance correction.
> * __Volatility:__ The standard deviation of the growth rates is $\sigma/\sqrt{\Delta t}$. We therefore need to account for the observation interval when estimating the GBM volatility $\sigma$ from their spread.

We will estimate $\mu_g$ and $\sigma$ from the growth-rate series, then
recover $\mu=\mu_g+\sigma^2/2$. Let's start with volatility.

### Volatility

Suppose we have $N+1$ price observations $\{S_{t_0},S_{t_1},\ldots,S_{t_N}\}$,
separated by $\Delta t$ years, with $N\geq2$. These give $N$ one-step growth
rates $\{g_1,\ldots,g_N\}$. Their sample mean $g^\prime$ and sample standard
deviation $\sigma_g$ are:

$$
\begin{aligned}
g^\prime&=\frac{1}{N}\sum_{j=1}^{N}g_j,\\
\sigma_g&=\underbrace{\sqrt{\frac{1}{N-1}\sum_{j=1}^{N}(g_j-g^\prime)^2}}_{\text{volatility of the growth rate}}.
\end{aligned}
$$

The denominator $N-1$ accounts for estimating one mean from the $N$ growth
observations.

> __GBM volatility:__ In L3a, we distinguished growth-rate volatility $\sigma_g$,
> interval log-return volatility $\sigma_r(\Delta t)$, and annualized log-return
> volatility $\sigma_{\mathrm{ann}}$. Under constant-parameter GBM, these give
> the volatility estimate:
> $$
> \boxed{
> \hat\sigma
> =\sqrt{\Delta t}\,\sigma_g
> =\frac{\sigma_r(\Delta t)}{\sqrt{\Delta t}}
> =\sigma_{\mathrm{ann}}.
> }
> $$
> With $\Delta t$ measured in years, $\hat\sigma$ has units of
> $\mathrm{yr}^{-1/2}$. The model's independent increments and constant volatility
> supply the assumptions behind the square-root-of-time scaling discussed in L3a.

For daily data, $\Delta t=1/252$ years, so we divide the standard deviation
of the annualized daily growth rates by $\sqrt{252}$. Equivalently, we can
multiply the standard deviation of daily log returns $g_j\Delta t$ by
$\sqrt{252}$. Both calculations give the same volatility estimate.

With an estimate of volatility, we can now estimate mean growth and recover
the GBM drift.

### Mean Growth and Drift

Next, let's estimate the mean growth rate $\mu_g$. Once we have an estimate
$\hat{\mu}_g$, we combine it with our volatility estimate to recover the GBM drift:

$$
\boxed{\hat{\mu}=\hat{\mu}_g+\frac{\hat{\sigma}^2}{2}.}
$$

Random price fluctuations can make mean growth difficult to estimate from a
finite historical sample. We will consider two methods:

> __Estimating the mean growth rate $\mu_g$:__
>
> * __Method 1: Average growth rate.__ Take the sample mean of the observed growth rates, giving $\hat{\mu}_g=g^\prime$. Under the constant-parameter GBM model, this is also the maximum likelihood estimate of $\mu_g$.
> * __Method 2: Linear regression.__ Fit a straight line to the logarithm of the share price as a function of time. The fitted slope estimates $\mu_g$. The log-price errors are correlated, so we must account for this when interpreting the uncertainty in the estimate.

Let's develop the linear-regression method and examine what the fitted line
tells us about mean growth.

### Linear Regression

Suppose we observe share prices $\{S_{t_0},S_{t_1},\ldots,S_{t_N}\}$ on the
grid $t_j=j\Delta t$. Taking the logarithm of the GBM solution gives:

$$
\ln(S_{t_j})=\ln(S_0)+\mu_g t_j+\sigma W(t_j),
\qquad j=0,1,\ldots,N.
$$

The Wiener process has mean zero, so the expected log price is:

$$
\mathbb E[\ln(S_{t_j})]=\ln(S_0)+\mu_g t_j.
$$

This is a straight line in time, with intercept $\ln(S_0)$ and slope $\mu_g$.
We therefore estimate mean growth by fitting a line to the observed log prices.

> __Interpreting the regression:__ The term $\sigma W(t_j)$ describes deviations
> from the expected log price. These errors are correlated across observation
> times, and their variance grows with time. We can estimate the slope using
> least squares; we will distinguish this error structure from the independent-error
> assumptions used to construct ordinary regression intervals below.

Let's write the regression as a system of equations and solve for the intercept
and slope.

Each of our $N+1$ log-price observations supplies an equation involving the
intercept and slope. Collecting these equations gives:

$$
\hat{\mathbf X}\boldsymbol{\theta}+\epsilon=\mathbf y,
$$

where $\boldsymbol{\theta}=[\ln(S_0),\,\mu_g]^\top$ contains the two parameters
and $\epsilon$ contains the errors. We fit both the intercept and slope using
all the observations.

The augmented design matrix contains a column of ones for the intercept and
a column of observation times:

$$
\hat{\mathbf X}=
\begin{bmatrix}
1&t_0\\
1&t_1\\
\vdots&\vdots\\
1&t_N
\end{bmatrix},
\qquad
\mathbf y=
\begin{bmatrix}
\ln(S_{t_0})\\
\ln(S_{t_1})\\
\vdots\\
\ln(S_{t_N})
\end{bmatrix}.
$$

Thus, $\hat{\mathbf X}$ has dimensions $(N+1)\times2$, while $\mathbf y$ has
$N+1$ entries. There are more observations than parameters, and the noisy
observations generally do not lie on a single straight line.

We choose the parameters that minimize the sum of squared differences between
observed and fitted log prices:

$$
\hat{\boldsymbol{\theta}}
=\underset{\boldsymbol{\theta}}{\operatorname{argmin}}\;
\left\|\mathbf y-\hat{\mathbf X}\boldsymbol{\theta}\right\|_2^2.
$$

> __Least-squares estimate:__ When $\hat{\mathbf X}$ has full column rank,
> the parameter estimate is:
> $$
> \boxed{
> \hat{\boldsymbol{\theta}}
> =(\hat{\mathbf X}^{\top}\hat{\mathbf X})^{-1}
> \hat{\mathbf X}^{\top}\mathbf y.
> }
> $$
> Its first component estimates the intercept, and its second component gives
> the mean-growth estimate $\hat{\mu}_g$.

Combining this slope estimate with the volatility estimate gives
$\hat{\mu}=\hat{\mu}_g+\hat{\sigma}^2/2$, completing the parameter estimates
needed for our GBM model.

### Regression Intervals Under Independent Errors

__How precisely have we estimated the intercept and mean growth?__ Let's introduce
the ordinary regression calculation used in the companion example. For this
calculation, suppose the errors are independent, normally distributed, and have
mean zero and common variance $\sigma_{\epsilon}^{2}$.

First, form the residuals $\mathbf r=\mathbf y-\hat{\mathbf X}\hat{\boldsymbol{\theta}}$.
With $n=N+1$ observations and $p=2$ fitted parameters, we estimate the error variance by:

$$
\hat{\sigma}_{\epsilon}^{2}=\frac{\|\mathbf r\|_2^2}{n-p}.
$$

The denominator accounts for the degrees of freedom used to fit the intercept
and slope. Combining this variance with the design matrix gives the estimated
standard error of each parameter:

$$
\mathrm{SE}(\hat{\theta}_j)=
\sqrt{\hat{\sigma}_{\epsilon}^{2}
\left[(\hat{\mathbf X}^{\top}\hat{\mathbf X})^{-1}\right]_{jj}},
\qquad j=1,2.
$$

The diagonal entries scale the residual variance for each parameter; taking
square roots expresses uncertainty in the parameter's own units.

> __Regression confidence interval:__ Under these independent normal-error
> assumptions, a confidence interval with level $1-\alpha$ for $\theta_j$ is:
> $$
> \hat{\theta}_j\pm t_{1-\alpha/2,\nu}\,\mathrm{SE}(\hat{\theta}_j),
> \qquad \nu=n-p.
> $$
> Here, $t_{1-\alpha/2,\nu}$ is the corresponding Student $t$ quantile, and
> $j=1,2$ selects the intercept or mean growth, respectively.

The [advanced drift-uncertainty example](advanced/drift-uncertainty/CHEME-5660-L4b-Advanced-DriftUncertainty-Fall-2026.ipynb) develops parameter uncertainty under
the GBM assumptions and examines its effect on target-return probabilities.
Let's apply the parameter estimates and regression interval formulas to historical
share-price data.

> __Example:__
>
> [▶ Estimate a single-asset GBM](CHEME-5660-L4b-Example-Parameters-SAGBM-Fall-2026.ipynb). We estimate mean growth and volatility, recover the GBM drift, and simulate possible price paths. We compare the fitted model's price distribution with an observed historical trajectory and examine the assumptions behind the comparison.

To assess how well the model describes observed price changes, let's compare
its predictions with the stylized facts introduced in L3a.

___


## Stylized Facts

In L3a, we examined three recurring patterns in observed growth rates: heavy
tails, little linear autocorrelation, and volatility clustering. Let's compare
these patterns with the predictions of our GBM model. For equally spaced,
non-overlapping intervals,

$$
g_j=\mu_g+\frac{\sigma}{\sqrt{\Delta t}}Z_j,
\qquad Z_j\overset{\mathrm{iid}}{\sim}\mathcal N(0,1).
$$

The parameters $\mu_g$ and $\sigma$ are constant, and the shocks are modeled
as independent across time.

> __Which stylized facts can the model reproduce?__
>
> * __Heavy-tailed growth rates:__ The model predicts normally distributed growth rates, so it assigns too little probability to the extreme movements observed in historical data. It does not reproduce heavy tails.
> * __Little linear autocorrelation:__ Independent shocks give zero covariance between growth rates at different steps:
>     $$
>     \operatorname{Cov}(g_j,g_{j+\tau})
>     =\frac{\sigma^2}{\Delta t}\operatorname{Cov}(Z_j,Z_{j+\tau})
>     =0,\qquad \tau\geq1.
>     $$
>     This is consistent with the weak linear dependence observed in signed growth rates, although independence is a stronger assumption than zero autocorrelation.
> * __Volatility clustering:__ Independent shocks and constant volatility give no mechanism for persistent dependence in the magnitudes of growth rates. A large absolute movement does not make another large movement more likely at the next step. The model therefore does not reproduce volatility clustering.

These comparisons explain which features of the data our model captures and
which it misses. Let's now use its analytical price distribution to calculate
the probability that a trade exceeds a target scaled NPV at a scheduled sale time.

___


## GBM Trade Rule

Let's return to the stock trade developed in L4a. We want to calculate the
probability that its scaled NPV exceeds a target at a scheduled sale time.

> __Scenario:__ Suppose we purchase $n_0>0$ shares of ticker `XYZ` at time $0$
> for $S_0>0$ USD/share. We sell all shares at time $T=N\Delta t$ for $S_T$
> USD/share, where $N\geq1$ and $\Delta t>0$ is measured in years. We assume
> no dividends, transaction fees, or bid–ask spread.

The purchase is a cash outflow, and the sale is a cash inflow. Let $g_y$ denote
the constant, continuously compounded benchmark growth rate, measured in inverse
years. Discounting the sale proceeds to time zero gives:

$$
\operatorname{NPV}(g_y,T)
=\underbrace{-n_0S_0}_{\text{purchase today}}
+\underbrace{n_0S_Te^{-g_yT}}_{\text{present value of sale proceeds}}.
$$

Dividing by the initial investment gives the __scaled NPV__:

$$
\boxed{
\rho_T
=\frac{\operatorname{NPV}(g_y,T)}{n_0S_0}
=\left(\frac{S_T}{S_0}\right)e^{-g_yT}-1.
}
$$

This dimensionless quantity measures the discounted fractional return on our
initial investment. A positive scaled NPV means that the present value of the
sale proceeds exceeds the purchase cost.

In L4a, we used a lattice to describe the uncertain sale price $S_T$. Let's now
substitute the GBM solution into this expression.

$$
\begin{aligned}
\rho_T
&=\exp\!\left[\mu_gT+\sigma\sqrt T\,Z\right]e^{-g_yT}-1\\
&=\exp\!\left[(\mu_g-g_y)T+\sigma\sqrt T\,Z\right]-1,
\qquad Z\sim\mathcal N(0,1).
\end{aligned}
$$

Discounting replaces $\mu_g$ with $\mu_g-g_y$ in the growth term, while the
uncertainty comes from the normally distributed shock $Z$.

> __Distribution of the scaled NPV:__ The quantity $1+\rho_T$ is lognormally
> distributed, so:
> $$
> \ln(1+\rho_T)
> \sim\mathcal N\!\left((\mu_g-g_y)T,\;\sigma^2T\right).
> $$
> The scaled NPV itself is a lognormal random variable shifted downward by one,
> and therefore satisfies $\rho_T>-1$.

For short holding periods with $|g_y|T\ll1$, the discount factor is
approximately one, giving $\rho_T\approx S_T/S_0-1$. We will retain the exact
discounted expression so our calculation also applies to longer holding periods.

Let's use this distribution to calculate the probability that $\rho_T$
exceeds a specified target.



### Terminal Target Probability

Choose a target scaled NPV $\rho_\star>-1$. We want the probability that
$\rho_T>\rho_\star$ at the scheduled sale time $T$. Adding one to both sides and taking the logarithm preserves the inequality:

$$
\rho_T>\rho_\star
\quad\Longleftrightarrow\quad
\ln(1+\rho_T)>\ln(1+\rho_\star).
$$

Substituting our expression for the log of $1+\rho_T$ gives:

$$
(\mu_g-g_y)T+\sigma\sqrt T\,Z>\ln(1+\rho_\star).
$$

Subtracting the mean and dividing by the positive standard deviation
$\sigma\sqrt T$ gives a threshold for the standard normal shock:

$$
Z>z_\star,
\qquad
z_\star=
\frac{\ln(1+\rho_\star)-(\mu_g-g_y)T}
{\sigma\sqrt T}.
$$

> __Terminal target probability:__ For $T>0$, $\sigma>0$, and $\rho_\star>-1$,
> the probability of exceeding the target is:
> $$
> \boxed{
> \mathbb P(\rho_T>\rho_\star)
> =\mathbb P(Z>z_\star)
> =1-\Phi(z_\star).
> }
> $$
> Here, $\Phi$ is the standard normal cumulative distribution function. It gives
> the probability of a shock at or below $z_\star$; subtracting from one gives
> the probability above the threshold.

For targets $\rho_\star\leq-1$, the probability is one because the scaled NPV
is always greater than $-1$ under this model. These probabilities concern the
scheduled sale time; selling when a boundary is first reached requires a
separate calculation.

Let's evaluate the target probability using mean growth and volatility
estimated from historical data.

> __Example:__
>
> [▶ Evaluate an NPV-based GBM trade rule](CHEME-5660-L4b-Example-GBM-NPV-TradeRule-Fall-2026.ipynb). We calculate the probability of exceeding a target scaled NPV for a selected firm and holding period. We check the calculation using the median return, which must be exceeded with probability one half, then plot how the probability changes as we raise the target.

The optional notebooks below extend this analysis by connecting the lattice
and continuous models, examining alternative exit rules, and investigating
parameter uncertainty and simulation methods.

___

## Optional Advanced Material
The notebooks below extend today's material. They are optional and are not prerequisites for L5a; the [advanced index](advanced/README.md) lists them with a suggested order.

* [▶ From the binomial lattice to GBM](advanced/lattice-limit/CHEME-5660-L4b-Advanced-LatticeToGBM-Fall-2026.ipynb). How does the binomial lattice connect to our continuous-time price model? We choose branch factors and probabilities consistent with the GBM parameters, then reduce the time step while keeping the holding period fixed. We examine how the terminal price distribution approaches a lognormal distribution and how the lattice target probability approaches the formula developed in this lecture.

* [▶ First-passage rules under GBM](advanced/first-passage/CHEME-5660-L4b-Advanced-FirstPassage-GBM-Fall-2026.ipynb). What changes if we sell when the price first reaches a take-profit or stop-loss boundary? We compare reaching a boundary before the sale date with finishing beyond it. We then calculate the probabilities of reaching either boundary first, or neither, using lattice calculations and simulation. We also examine how often we must check the price to detect these events.

* [▶ Uncertainty in mean growth](advanced/drift-uncertainty/CHEME-5660-L4b-Advanced-DriftUncertainty-Fall-2026.ipynb). Why can mean growth remain uncertain even with thousands of price observations? We derive how the precision of the sample-mean estimate depends on the length of the observation period and compare this with volatility estimation. We then examine how uncertainty in mean growth changes the probability of exceeding a target scaled NPV.

* [▶ Monte Carlo versus the closed form](advanced/monte-carlo/CHEME-5660-L4b-Advanced-MonteCarlo-TargetProbability-Fall-2026.ipynb). How many simulated price paths do we need to estimate a target probability accurately? We compare simulation estimates and their standard errors with the analytical result. We also compare the exact GBM transition with a numerical approximation and test paired paths with opposite shocks, called antithetic variates, to reduce sampling error.

___


## Summary

In this lecture, we introduced a continuous-time model of share prices,
developed methods for estimating its parameters, and used its price
distribution to evaluate a stock trade.

> __Key Takeaways__
>
> * __Continuous-time price modeling:__ We introduced geometric Brownian motion and used its exact solution to describe future share prices and develop a one-step simulation procedure. The model gives normally distributed log returns and lognormally distributed prices. Comparing its predictions with the stylized facts showed that it captures weak linear autocorrelation but misses heavy tails and volatility clustering.
> * __Mean growth, drift, and volatility:__ We developed estimates of mean growth using sample averages and log-price regression, and connected growth-rate dispersion to the GBM volatility parameter. We then recovered the drift using the half-variance correction, $\mu=\mu_g+\sigma^2/2$. This connected the statistics of observed growth rates to the parameters needed for our price model.
> * __Probability of exceeding a terminal target:__ We substituted the GBM price solution into the scaled NPV of a long stock position. Taking the logarithm and standardizing converted the return target into a standard-normal threshold, allowing us to calculate its exceedance probability directly. The result accounts for the holding period and benchmark growth rate and evaluates the trade at its scheduled sale time.

Next time, we extend the model to several correlated assets and introduce the
covariance structure needed for portfolios.

___


## Disclaimer and Risks

__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products, or any investment or trading advice or strategy, is made, given, or endorsed by the teaching team.

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.
